# 第 3 章: 決定木の探索と可視化

深さによる正解率の変化、決定木の構造、Tribuo との予測の違い、特徴量の重要度を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)
@file:DependsOn("org.tribuo:tribuo-classification-tree:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter01.accuracy
import chapter02.prepareIris
import chapter03.DecisionTree
import chapter03.formatTree
import chapter03.predictWithTribuo
import chapter03.trainTribuoTree
import org.tribuo.classification.Label
import org.tribuo.common.tree.TreeModel
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val irisCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "iris.csv")
val split = prepareIris(irisCsv, testSize = 0.3, seed = 0)

## 深さと正解率

In [ ]:
val depths = (1..7).toList()
val models = depths.map { DecisionTree(it).fit(split.xTrain, split.tTrain) }
val scores =
    dataFrameOf(
        "深さ" to depths + depths,
        "データ" to depths.map { "訓練データ" } + depths.map { "テストデータ" },
        "正解率" to models.map { accuracy(it.predict(split.xTrain), split.tTrain) } +
            models.map { accuracy(it.predict(split.xTest), split.tTest) },
    )
scores.plot {
    line {
        x("深さ")
        y("正解率")
        color("データ")
    }
    points {
        x("深さ")
        y("正解率")
        color("データ")
    }
    layout.title = "決定木の深さと正解率"
}

In [ ]:
scores

## 決定木の構造

In [ ]:
println(formatTree(checkNotNull(DecisionTree(2).fit(split.xTrain, split.tTrain).tree)))

## Tribuo との突き合わせ

In [ ]:
dataFrameOf(
    "深さ" to depths,
    "予測が違う件数" to depths.map { depth ->
        val mine = DecisionTree(depth).fit(split.xTrain, split.tTrain).predict(split.xTest)
        val tribuo = predictWithTribuo(trainTribuoTree(split.xTrain, split.tTrain, depth, 1.0f), split.xTest)
        mine.zip(tribuo).count { (a, b) -> a != b }
    },
)

## 特徴量の重要度

Tribuo の決定木の `getTopFeatures` は、特徴量が分割に使われた回数を返す。scikit-learn の `feature_importances_`（不純度の減少量）とは別の尺度なので、深さ 3 の木で自作の決定木の構造と見比べる。

In [ ]:
val tribuoTree = trainTribuoTree(split.xTrain, split.tTrain, 3, 1.0f) as TreeModel<Label>
tribuoTree.getTopFeatures(-1)

In [ ]:
println(formatTree(checkNotNull(DecisionTree(3).fit(split.xTrain, split.tTrain).tree)))